# Lorenz-style 20,000 bins (MWI 1997 and BEL 2000)

This notebook adapts the Lorenz-curve style pipeline: check monotonic welfare shares, optionally duplicate/split high-weight outliers, create 20,000 weighted bins, and compute mean welfare after grouping.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('.').resolve()
PROJECT_ROOT = next(
    (p for p in [ROOT, *ROOT.parents] if (p / '.git').exists() or (p / 'pyproject.toml').exists()),
    ROOT
)
TARGET_DIR = PROJECT_ROOT / 'target_file'
TARGET_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT

WindowsPath('C:/Users/wb661551/OneDrive - WBG/Desktop/Internship/Bottom Censoring/bindata_check')

In [2]:
def weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    sw = weights.sum()
    if sw <= 0:
        return np.nan
    return float(np.sum(values * weights) / sw)


def new_bins(welfare, weight, nbins=100, tolerance=1e-6, ids=None):
    welfare = np.asarray(welfare, dtype=float)
    weight = np.asarray(weight, dtype=float)
    if welfare.shape[0] != weight.shape[0]:
        raise ValueError('welfare and weight must have the same length')

    valid = (~np.isnan(welfare)) & (~np.isnan(weight)) & (weight > 0)
    welfare = welfare[valid]
    weight = weight[valid]

    if ids is None:
        ids = np.arange(1, welfare.shape[0] + 1, dtype=int)
    else:
        ids = np.asarray(ids)[valid]

    order = np.argsort(welfare, kind='mergesort')
    welfare = welfare[order]
    weight = weight[order]
    ids = ids[order]

    total_weight = float(weight.sum())
    if total_weight <= 0:
        return pd.DataFrame(columns=['id', 'bin', 'weight', 'welfare'])

    bin_size = total_weight / nbins

    out_id = []
    out_bin = []
    out_weight = []
    out_welfare = []

    cur_bin = 1
    cur_weight = 0.0

    for i in range(welfare.shape[0]):
        wlf = float(welfare[i])
        wt = float(weight[i])
        id_i = int(ids[i])

        while wt > 0 and cur_bin <= nbins:
            room = bin_size - cur_weight
            if abs(room) < tolerance:
                cur_bin += 1
                cur_weight = 0.0
                continue

            take = wt if wt < room else room

            out_id.append(id_i)
            out_bin.append(cur_bin)
            out_weight.append(take)
            out_welfare.append(wlf)

            wt -= take
            cur_weight += take

            if cur_weight >= (bin_size - tolerance):
                cur_bin += 1
                cur_weight = 0.0

    return pd.DataFrame(
        {
            'id': out_id,
            'bin': out_bin,
            'weight': out_weight,
            'welfare': out_welfare,
        }
    )


def lorenz_table(df, nq=100, tolerance=1e-6):
    req = {'welfare', 'weight'}
    miss = req - set(df.columns)
    if miss:
        raise ValueError(f'Missing required columns: {sorted(miss)}')

    x = df[['welfare', 'weight']].copy()
    if 'reporting_level' in df.columns:
        x['reporting_level'] = df['reporting_level'].astype(str).fillna('national')
    else:
        x['reporting_level'] = 'national'

    x = x.dropna(subset=['welfare', 'weight'])
    x = x[x['weight'] > 0].copy()

    if x.empty:
        return pd.DataFrame(
            columns=[
                'reporting_level', 'bin', 'avg_welfare', 'pop_share',
                'welfare_share', 'quantile', 'pop', 'cum_pop_share', 'cum_welfare_share'
            ]
        )

    if x['reporting_level'].nunique() > 1:
        x2 = x.copy()
        x2['reporting_level'] = 'national'
        x = pd.concat([x, x2], ignore_index=True)

    out_parts = []
    for rl, sub in x.groupby('reporting_level', sort=False):
        split = new_bins(
            welfare=sub['welfare'].to_numpy(dtype=float),
            weight=sub['weight'].to_numpy(dtype=float),
            nbins=nq,
            tolerance=tolerance,
            ids=np.arange(1, len(sub) + 1, dtype=int)
        )

        if split.empty:
            continue

        split['wt_welfare'] = split['welfare'] * split['weight']
        tot_pop = float(split['weight'].sum())
        tot_wlf = float(split['wt_welfare'].sum())

        g = split.groupby('bin', as_index=False).agg(
            pop=('weight', 'sum'),
            wt_welfare=('wt_welfare', 'sum'),
            quantile=('welfare', 'max')
        )
        g['avg_welfare'] = np.where(g['pop'] > 0, g['wt_welfare'] / g['pop'], np.nan)
        g['pop_share'] = np.where(tot_pop > 0, g['pop'] / tot_pop, np.nan)
        g['welfare_share'] = np.where(tot_wlf > 0, g['wt_welfare'] / tot_wlf, np.nan)
        g = g[['bin', 'avg_welfare', 'pop_share', 'welfare_share', 'quantile', 'pop']]
        g.insert(0, 'reporting_level', rl)
        g = g.sort_values('bin').reset_index(drop=True)
        g['cum_pop_share'] = g['pop_share'].cumsum()
        g['cum_welfare_share'] = g['welfare_share'].cumsum()
        out_parts.append(g)

    if not out_parts:
        return pd.DataFrame()

    return pd.concat(out_parts, ignore_index=True)


def welfare_share_is_monotone(lt, tol=1e-12):
    if lt.empty:
        return True
    for _, g in lt.groupby('reporting_level', sort=False):
        d = g['welfare_share'].diff().dropna()
        if (d < -tol).any():
            return False
    return True


def find_outliers_by_weight(df, weight_col='weight', threshold=2.5):
    out = df.copy()
    mean_w = out[weight_col].mean()
    sd_w = out[weight_col].std(ddof=1)
    cutoff = mean_w + threshold * sd_w
    out['is_outlier'] = out[weight_col] > cutoff
    return out


def optimize_ratio(y, m):
    y = np.asarray(y, dtype=float)
    opt_x = np.maximum(m, np.sqrt(np.maximum(y, 0)))
    to_one = opt_x > (y / 2)
    opt_x[to_one] = 1.0
    rep = np.rint(np.divide(y, opt_x, out=np.ones_like(y), where=opt_x > 0)).astype(int)
    rep[rep < 1] = 1
    return rep


def duplicate_obs(df, weight_col='weight'):
    x = df.copy()
    min_w = float(x[weight_col].min())
    x['hhindex'] = np.arange(1, len(x) + 1, dtype=int)
    rep_count = optimize_ratio(x[weight_col].to_numpy(dtype=float), min_w)
    rep_count = np.where(x['is_outlier'].to_numpy(), rep_count, 1)
    x['rep_count'] = rep_count.astype(int)
    y = x.loc[np.repeat(np.arange(len(x)), x['rep_count'].to_numpy())].copy()
    return y.reset_index(drop=True)


def add_new_weights(df, weight_col='weight'):
    x = df.copy()
    mask = x['is_outlier'].to_numpy()
    x.loc[mask, weight_col] = x.loc[mask, weight_col] / x.loc[mask, 'rep_count']
    return x


def duplicate_households_official(df, weight_col='weight', threshold=2.5, li=5, super_limit=20, nq_check=100):
    r = df.copy()
    i = 0
    threshold_cur = float(threshold)
    li_cur = int(li)

    while True:
        lt = lorenz_table(r, nq=nq_check)
        welfare_ok = welfare_share_is_monotone(lt)
        if welfare_ok or i >= super_limit:
            meta = {
                'welfare_share_OK': bool(welfare_ok),
                'threshold': threshold_cur,
                'iterations': i,
                'lorenz': lt,
            }
            return r, meta

        i += 1
        t = find_outliers_by_weight(r, weight_col=weight_col, threshold=threshold_cur)
        t = duplicate_obs(t, weight_col=weight_col)
        t = add_new_weights(t, weight_col=weight_col)

        keep_cols = [c for c in df.columns if c in t.columns]
        r = t[keep_cols].copy()

        if (not welfare_ok) and (i >= li_cur) and (threshold_cur > 0):
            threshold_cur = max(threshold_cur - 0.5, 0.0)
            li_cur = li_cur * 2


def build_20000_with_lorenz(country_file, tag, nq=20000):
    path = PROJECT_ROOT / '01-input' / 'country' / country_file
    if not path.exists():
        raise FileNotFoundError(f'Input not found: {path}')

    dt = pd.read_stata(path, convert_categoricals=False)
    needed = ['welfare', 'weight']
    missing = [c for c in needed if c not in dt.columns]
    if missing:
        raise ValueError(f'{country_file} missing columns: {missing}')

    d = dt[needed].dropna().copy()
    d = d[d['weight'] > 0].copy()

    if 'cpi2021' in dt.columns and 'icp2021' in dt.columns:
        cpi = dt['cpi2021'].dropna()
        icp = dt['icp2021'].dropna()
        if not cpi.empty and not icp.empty:
            d['welfare'] = d['welfare'] / (float(cpi.iloc[0]) * float(icp.iloc[0]) * 365.0)

    if 'reporting_level' in dt.columns:
        d['reporting_level'] = dt.loc[d.index, 'reporting_level'].astype(str).fillna('national')
    else:
        d['reporting_level'] = 'national'

    adjusted, meta = duplicate_households_official(
        d, weight_col='weight', threshold=2.5, li=5, super_limit=20, nq_check=100
    )

    lt_20000 = lorenz_table(adjusted, nq=nq)
    lt_national = lt_20000[lt_20000['reporting_level'] == 'national'].copy()
    mean_after_20000 = weighted_mean(lt_national['avg_welfare'], lt_national['pop'])

    out_csv = TARGET_DIR / f'{tag}_20000_bins_lorenz.csv'
    lt_national.to_csv(out_csv, index=False)

    return {
        'country': tag,
        'rows_in_adjusted_microdata': int(len(adjusted)),
        'iterations': int(meta['iterations']),
        'welfare_share_OK': bool(meta['welfare_share_OK']),
        'mean_welfare_after_20000': float(mean_after_20000),
        'output_csv': str(out_csv),
        'table': lt_national
    }

In [3]:
res_mwi = build_20000_with_lorenz('MWI_1997.dta', 'MWI_1997', nq=20000)
res_mwi['country'], res_mwi['iterations'], res_mwi['welfare_share_OK'], res_mwi['mean_welfare_after_20000'], res_mwi['output_csv']

('MWI_1997',
 0,
 True,
 5.779647809449764,
 'C:\\Users\\wb661551\\OneDrive - WBG\\Desktop\\Internship\\Bottom Censoring\\bindata_check\\target_file\\MWI_1997_20000_bins_lorenz.csv')

In [4]:
res_bel = build_20000_with_lorenz('BEL_2000.dta', 'BEL_2000', nq=20000)
res_bel['country'], res_bel['iterations'], res_bel['welfare_share_OK'], res_bel['mean_welfare_after_20000'], res_bel['output_csv']

('BEL_2000',
 0,
 True,
 63.428920445960706,
 'C:\\Users\\wb661551\\OneDrive - WBG\\Desktop\\Internship\\Bottom Censoring\\bindata_check\\target_file\\BEL_2000_20000_bins_lorenz.csv')

In [5]:
summary = pd.DataFrame([
    {k: v for k, v in res_mwi.items() if k != 'table'},
    {k: v for k, v in res_bel.items() if k != 'table'}
])
summary

,country,rows_in_adjusted_microdata,iterations,welfare_share_OK,mean_welfare_after_20000,output_csv
0,MWI_1997,10698,0,True,5.779648,C:\Users\wb661551\OneDrive - WBG\Desktop\Inter...
1,BEL_2000,1000,0,True,63.428920,C:\Users\wb661551\OneDrive - WBG\Desktop\Inter...
